# **Fine-Tune Pre-Trained Classification Model With QLORA**

### **Import Libraries**

In [26]:
import torch
import numpy as np
import torch.nn as nn   
import json
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, TaskType,get_peft_model, replace_lora_weights_loftq
from transformers import TrainingArguments, Trainer

In [2]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
device.type

'cuda'

In [3]:
def save_data_to_file (data, file_path):
    
    with open(file_path, "w") as file:
        json.dump(data, file,indent=4)
    print(f"file created successfully at {file_path}")

In [4]:
def load_data_from_file(file_path):
    
    with open(file_path, "r") as file:
        load_file = json.load(file)
    
    return load_file

### **Load Dataset**

In [5]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [6]:
train_labels = dataset['train']['label']
num_classes = len(set(train_labels))
num_classes

2

In [7]:
imdb_labels = {0: 'negative review', 1: 'positive review'}

In [8]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

##### Tokenizer Map Function

In [9]:
def tokenized_text (data):
    return tokenizer(data['text'], 
                     padding='max_length', max_length=512, truncation=True)

In [10]:
dataset = dataset.map(tokenized_text, batched=True)

In [11]:
dataset['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [12]:
dataset = dataset.rename_column('label', 'labels')

In [13]:
dataset.set_format(type='torch', columns = ['labels', 'input_ids', 'attention_mask'])

In [14]:
dataset['train'][0]

{'labels': tensor(0),
 'input_ids': tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
          2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
          2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
          2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
          1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
          2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
          6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
          1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
          5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
         14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
          1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
          2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
         25430, 1

In [15]:
dataset_train = dataset['train']
dataset_test = dataset['test']

### **BitsAndBytes Configuration**

In [16]:
config_bits = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype= torch.bfloat16,
    llm_int8_skip_modules=['classifier', 'pre_classifier']
)

In [17]:
#ids to labels
id_to_label = {0: "Negative", 1: "Positive"}

# labels to ids
label_to_id = dict((v,k) for k,v in id_to_label.items())

### **Define Model Inatance**

In [18]:
qlora_model = AutoModelForSequenceClassification.from_pretrained(
                                                "distilbert-base-uncased",
                                                id2label = id_to_label,
                                                label2id = label_to_id,
                                                num_labels = num_classes,
                                                quantization_config = config_bits
)

W0514 19:00:22.093000 1412 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   4%|▍         | 4/100 [00:00<00:02, 38.45it/s]C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 316.46it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bia

After loading the model with 4-bit or 8-bit quantization, the model is only quantized for inference/memory efficiency. It is not yet configured properly for stable training.

This step prepares it for QLoRA fine-tuning. Internally, PEFT modifies the model for low-bit training by:

* freezing the original pretrained weights
* enabling gradients only where needed
* converting sensitive layers (like layer norms) to higher precision (float32)
* improving numerical stability during backpropagation
* optionally enabling gradient checkpointing support

In [19]:
qlora_model = prepare_model_for_kbit_training(qlora_model)

#### **Define PERF Model**

In [20]:
# define lora config
lora_config = LoraConfig(
    task_type= TaskType.SEQ_CLS,
    r = 8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q_lin', 'k_lin', 'v_lin']
)

peft_model = get_peft_model(qlora_model, lora_config)

peft_model_qlora is now configured as a QLoRA model and ready for training. Before starting the training process, an additional optimization step will be applied by reinitializing the LoRA weights with LoftQ, a technique that has demonstrated improved performance for quantized model training.

In [21]:
replace_lora_weights_loftq(peft_model)

In [22]:
print(peft_model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  

In [23]:
peft_model.print_trainable_parameters()

trainable params: 813,314 || all params: 67,768,324 || trainable%: 1.2001


### **Define Model Training** 

In [24]:
training_args = TrainingArguments(
    "./result_qlora",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    weight_decay=0.1,
    logging_strategy="steps",
    logging_steps=100,
    fp16=True
)

In [27]:
def compute_accuracy (predict_eval):
    
    load_accuracy = evaluate.load("accuracy")
    
    logits, labels = predict_eval
    
    prediction = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=prediction, references=labels)['accuracy']
    
    return {"accuracy": accuracy}

In [ ]:
trainer = Trainer(
    model= peft_model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    tokenizer = tokenizer,
   
)